Since the extraction time for the vision modality is relatively long, the vision modality is extracted separately, while the audio and text modalities are combined and extracted together. After extraction, the features are merged into a single pkl file.

1.Image Feature Extraction

In [ ]:
import pandas as pd
import torch
from torchvision import models, transforms
from PIL import Image
import os
import pickle
import torch.nn as nn
import numpy as np
from torchvision.models import ResNet50_Weights, ResNet101_Weights

# Set device to GPU (if available) or CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define the ResNet model for feature extraction
class ResNetFeatureExtractor(nn.Module):
    def __init__(self, model_name='resnet50', weights=ResNet50_Weights.IMAGENET1K_V1):
        super(ResNetFeatureExtractor, self).__init__()
        if model_name == 'resnet50':
            self.model = models.resnet50(weights=weights)
        elif model_name == 'resnet101':
            self.model = models.resnet101(weights=ResNet101_Weights.IMAGENET1K_V1)
        else:
            raise ValueError("Unsupported model type")

        # Remove the final classification layer
        self.features = nn.Sequential(*list(self.model.children())[:-1])

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return x

# Image preprocessing steps
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Read and preprocess the image
def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = preprocess(image)
    image = image.unsqueeze(0)  # Add batch dimension
    return image

# Extract image features
def extract_image_features(image_paths, model_name='resnet50'):
    model = ResNetFeatureExtractor(model_name=model_name).to(device)
    model.eval()

    features_list = []
    for image_path in image_paths:
        image = load_image(image_path).to(device)
        with torch.no_grad():
            features = model(image)
        features_list.append(features.cpu().numpy())

    # Compress all features into a (N, 2048) matrix
    return np.vstack(features_list)

# Data loading function
def load_data(file):
    data = pd.read_csv(file)
    return data['video_id'], data['clip_id'], data['text'], data['label'], data['annotation'], data['mode']

# Data saving function
def save_features(data, file_name):
    with open(file_name, 'wb') as f:
        pickle.dump(data, f)

# Load the label file
video_path = '/root/autodl-tmp/MOSEI_test/mosei_move/Raw'
video_ids, clip_ids, texts, labels, annotations, modes = load_data('/root/autodl-tmp/MOSEI_test/train_data.csv')

features = []
errors = []

# Iterate over the label file
for video_id, clip_id, text, label, annotation, mode in zip(video_ids, clip_ids, texts, labels, annotations, modes):
    clip_id_ = str(clip_id)

    try:
        # Extract image features
        image_folder_path = os.path.join(video_path, video_id, clip_id_)
        image_files = [os.path.join(image_folder_path, img) for img in os.listdir(image_folder_path) if img.endswith('.png')]
        
        if not image_files:
            print(f"No images found in {image_folder_path}.")
            errors.append({
                'video_id': video_id,
                'clip_id': clip_id,
                'error': 'No images found'
            })
            continue  # Skip this iteration if no image files are found

        # Extract features from each image and stack them into a (N, 2048) matrix
        vision_features = extract_image_features(image_files)
        
        # Save the image feature matrix and other information into the list
        features.append({
            'video_id': video_id,
            'clip_id': clip_id,
            'text': text,
            'label_1': label,  # Add label as label_1
            'label': annotation,
            'mode': mode,
            'vision_features': vision_features  # Save the compressed feature matrix (N, 2048)
        })

    except Exception as e:
        print(f"Error processing video_id: {video_id}, clip_id: {clip_id}. Error: {e}")
        errors.append({
            'video_id': video_id,
            'clip_id': clip_id,
            'error': str(e)
        })

# Save the features
save_features(features, 'train_vision_only_MOSEI_features_with_labels.pkl')

# Save the error log
save_features(errors, 'train_vision_only_MOSEI_feature_extraction_errors.pkl')


Text Feature Extraction and Audio Feature Extraction

In [ ]:
import pandas as pd
import torch
import torchaudio
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Wav2Vec2Processor, Wav2Vec2Model
import os
import pickle
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
from transformers import HubertModel

# Set device to GPU (if available) or CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load text model and tokenizer
model_path = '/root/autodl-tmp/features_1/sbert'
tokenizer = AutoTokenizer.from_pretrained(model_path)
text_model = AutoModel.from_pretrained(model_path).to(device)

# Load audio model and processor
audio_model_path = '/root/autodl-tmp/features_1/Hubert'
audio_processor = Wav2Vec2Processor.from_pretrained(audio_model_path)
audio_model = HubertModel.from_pretrained(audio_model_path).to(device)

# Data loading function
def load_data(file):
    data = pd.read_csv(file)
    return data['video_id'], data['clip_id'], data['text'], data['label'], data['annotation'], data['mode']

# Data saving function
def save_features(data, file_name):
    with open(file_name, 'wb') as f:
        pickle.dump(data, f)

# Function to get speech features
def get_speech_feature(file_path):
    waveform, sample_rate = torchaudio.load(file_path)
    # Ensure the audio is mono (single channel)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Resample to 16000Hz
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)

    inputs = audio_processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}  # Move input data to GPU
    with torch.no_grad():
        outputs = audio_model(**inputs)

    feature = outputs.last_hidden_state.cpu()  # Move features to CPU
    return feature

video_path = '/root/autodl-tmp/MOSEI_test/Raw/'
video_ids, clip_ids, texts, labels, annotations, modes = load_data('/root/autodl-tmp/MOSEI_test/train_combined_file.csv')

features = []
errors = []

for video_id, clip_id, text, label, annotation, mode in zip(video_ids, clip_ids, texts, labels, annotations, modes):
    clip_id_ = str(clip_id)
    file_path = os.path.join(video_path, video_id, f"{clip_id_}.mp3")

    try:
        # Extract text features
        text_inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        text_inputs = {key: val.to(device) for key, val in text_inputs.items()}  # Move input data to GPU
        with torch.no_grad():
            text_outputs = text_model(**text_inputs)
        text_feature = text_outputs.last_hidden_state.cpu().numpy()  # Move features to CPU

        # Extract audio features
        if os.path.exists(file_path):  # Ensure the file exists
            audio_feature = get_speech_feature(file_path)
        else:
            print(f"File {file_path} not found.")
            errors.append({
                'video_id': video_id,
                'clip_id': clip_id,
                'error': 'File not found'
            })
            continue  # Skip this iteration if the file doesn't exist

        # Save features and other information to the list
        features.append({
            'video_id': video_id,
            'clip_id': clip_id,
            'text': text,
            'label_1': label,  # Add label as label_1
            'label': annotation,
            'mode': mode,
            'text_feature': text_feature,
            'audio_feature': audio_feature
        })

    except Exception as e:
        print(f"Error processing video_id: {video_id}, clip_id: {clip_id}. Error: {e}")
        errors.append({
            'video_id': video_id,
            'clip_id': clip_id,
            'error': str(e)
        })

# Save features
save_features(features, 'train_MOSEI_short_sbert_Hubert_long_features_with_labels.pkl')

# Save error log
save_features(errors, 'train_MOSEI_feature_extraction_errors.pkl')